### Setup pour Notebook NB02
Ce notebook :
- Récupérer la valeur du widget full_path (du notebook parent NB03) et créer un dataframe à partir du fichier CSV
- Créer un volume avec les valeurs des widgets (catalog, schema, volume) du notebook parent (NB03)
- Charge le fichier dans volume créer
- Charge les données brut dans une table {filename}


In [0]:
import pandas as pd

full_path_value = dbutils.widgets.get("full_path")
print(f"création du dataframe à partir du fichier {full_path_value}")

df = pd.read_csv(full_path_value)

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
filename = dbutils.widgets.get("filename")

# catalog = "workspace"
# schema = "default"
# volume = "spark_training"

print(f"Création du volume : {catalog}.{schema}.{volume} si il n'existe pas")

spark.sql(f"""
          CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}
          COMMENT 'Volume créé via Python : {catalog}.{schema}.{volume}'""")

In [0]:
print(f"Lecture du fichier : {full_path}")
df = pd.read_csv(full_path)

path = f"/Volumes/{catalog}/{schema}/{volume}/{filename}"
print(f"Ecriture du fichier dans le volume : {path}")

dbutils.fs.put(path, df.to_csv(index=False), overwrite=True)
print("Ecriture du fichier OK")

print(f"Création de la table : {filename.split(".")[0]}")

df_spark = spark.createDataFrame(df)

print(f"Création de la table {filename.split(".")[0]} OK")

spark.sql(f"DROP TABLE IF EXISTS {filename.split('.')[0]}")

df_spark.write \
    .mode("overwrite") \
    .saveAsTable(filename.split(".")[0])
print(f"Création de la table : {filename.split('.')[0]} OK")
